# Notebook 2 — Mechanism Dependence: Does It Matter How Practice Operates?

Notebook 0 established the formal properties Berman's SAW model requires.
Notebook 1 showed that evolutionary practice (ABM) produces binary weights
that pass three of five criteria but fail on cardinal variation and the Hart
advantage. This notebook asks: do different practice mechanisms produce
different weight structures — and does any single mechanism satisfy all five
theoretical requirements?

**Three mechanisms tested:**
- **ABM (evolutionary selection):** Agents succeed or fail; unsuccessful agents
  are replaced by mutated copies of successful ones. Models cultural evolution
  in legal practice.
- **MARL (strategic optimisation):** Neural network agents learn optimal
  invocation strategies through policy gradient. Models rational
  consensus-seeking.
- **Hybrid:** Neural learning with periodic evolutionary replacement. Tests
  whether combining mechanisms helps or whether they interfere.

**The prediction from Notebook 1:** ABM produces differentiation but not
gradation. If MARL produces gradation but not differentiation, the two
mechanisms have complementary failure patterns — and the paper's theoretical
contribution is that Berman's framework requires specifying *which kind*
of practice generates parameters.

**Hardware:** This notebook uses PyTorch for MARL. Select T4 GPU in Colab
runtime settings for 10–50× speedup on the neural network training.

**Runtime:** approximately 30–45 minutes on T4 GPU.

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%matplotlib inline
!pip install numpy pandas matplotlib seaborn scipy scikit-learn torch -q
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
print('Dependencies installed.')

Device: cuda
  GPU: Tesla T4
Dependencies installed.


## Core Model and ABM (from Notebooks 0 and 1)

Copied verbatim so this notebook runs independently.

In [3]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional
from collections import defaultdict
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
plt.rcParams.update({
    'font.size': 10, 'axes.titlesize': 12, 'axes.labelsize': 11,
    'figure.dpi': 100, 'savefig.dpi': 150, 'savefig.bbox': 'tight',
})

# --- Notebook 0 core ---
@dataclass
class Principle:
    name: str
    weight: float
    description: str = ""
    category: str = "general"
    def __post_init__(self):
        assert 0 <= self.weight <= 1

@dataclass
class Case:
    name: str
    facts: Dict[str, float]

class BermanModel:
    def __init__(self, principles, threshold=0.2):
        self.principles = {p.name: p for p in principles}
        self.threshold = threshold
    def net_force(self, case, weight_noise=0.0):
        f = 0.0
        for name, act in case.facts.items():
            if name not in self.principles: continue
            w = self.principles[name].weight
            if weight_noise > 0:
                w = np.clip(w + np.random.normal(0, weight_noise), 0, 1)
            f += w * act
        return f
    def is_determinate(self, case, weight_noise=0.0):
        return abs(self.net_force(case, weight_noise)) >= self.threshold

def make_principles(n, distribution, seed=0):
    rng = np.random.RandomState(seed)
    if distribution == 'uniform': weights = rng.uniform(0.60, 0.70, n)
    elif distribution == 'moderate': weights = rng.uniform(0.50, 0.80, n)
    elif distribution == 'high': weights = rng.uniform(0.30, 0.95, n)
    else: raise ValueError(distribution)
    return [Principle(f"P{i}", float(w)) for i, w in enumerate(weights)]

def bootstrap_ci(values, n_boot=2000, ci=95):
    arr = np.array(values, dtype=float)
    means = [np.mean(np.random.choice(arr, len(arr), replace=True)) for _ in range(n_boot)]
    lo = np.percentile(means, (100-ci)/2)
    hi = np.percentile(means, 100-(100-ci)/2)
    return float(np.mean(arr)), float(lo), float(hi)

def find_crossing(curve, levels, threshold):
    for j in range(1, len(curve)):
        if curve[j] < threshold and curve[j-1] >= threshold:
            frac = (threshold - curve[j-1]) / (curve[j] - curve[j-1])
            return float(levels[j-1] + frac * (levels[j] - levels[j-1]))
    return float(levels[-1])

# --- Notebook 1 ABM core ---
PRINCIPLES_DEFAULT = ['text','original','structure','precedent',
                      'dignity','equality','liberty','federalism','pragmatic']
METHODOLOGIES = ['textualist','purposivist','pragmatist','originalist']

def make_invocation_tendencies(spread='wide'):
    if spread == 'wide': hi, lo, md_r = (0.85,0.95),(0.02,0.08),(0.35,0.65)
    elif spread == 'medium': hi, lo, md_r = (0.70,0.80),(0.15,0.25),(0.40,0.60)
    elif spread == 'narrow': hi, lo, md_r = (0.60,0.70),(0.30,0.40),(0.42,0.58)
    else: raise ValueError(spread)
    structure = {
        'textualist':  {'high':['text','original'],'med':['structure','federalism','precedent'],'low':['dignity','equality','liberty','pragmatic']},
        'purposivist': {'high':['dignity','equality','liberty'],'med':['precedent','pragmatic','structure'],'low':['text','original','federalism']},
        'pragmatist':  {'high':['pragmatic','precedent'],'med':['structure','liberty','federalism','text'],'low':['dignity','equality','original']},
        'originalist': {'high':['original','text','federalism'],'med':['structure','precedent'],'low':['dignity','equality','liberty','pragmatic']},
    }
    tendencies = {}
    for method, groups in structure.items():
        t = {}
        for p in PRINCIPLES_DEFAULT:
            if p in groups['high']: t[p] = np.random.uniform(*hi)
            elif p in groups['low']: t[p] = np.random.uniform(*lo)
            else: t[p] = np.random.uniform(*md_r)
        tendencies[method] = t
    return tendencies

INVOCATION_TENDENCIES = make_invocation_tendencies('wide')

@dataclass
class Agent:
    agent_id: int
    methodology: str
    invocation_tendencies: Dict[str,float] = field(default_factory=dict)
    success_count: int = 0
    total_decisions: int = 0
    @classmethod
    def create(cls, agent_id, methodology, tendencies_map):
        base = tendencies_map[methodology].copy()
        jitter = {p: np.clip(v+np.random.normal(0,0.03),0.02,0.98) for p,v in base.items()}
        return cls(agent_id=agent_id, methodology=methodology, invocation_tendencies=jitter)
    def decide_case(self, activations, system_weights):
        invoked = []; net_force = 0.0
        for p, act in activations.items():
            base = self.invocation_tendencies.get(p,0.5)
            sw = system_weights.get(p,0.5)
            prob = 0.85*base + 0.15*sw
            if abs(act) > 0.3: prob = min(prob*1.15, 0.99)
            if np.random.random() < prob:
                invoked.append(p); net_force += sw*act
        return invoked, net_force
    def update_tendencies(self, invoked, was_successful, lr=0.04):
        if len(invoked) > 4: lr *= 0.5
        for p in self.invocation_tendencies:
            delta = lr if (p in invoked and was_successful) else (-lr if p in invoked else -lr*0.08)
            self.invocation_tendencies[p] = np.clip(self.invocation_tendencies[p]+delta, 0.02, 0.98)

class LegalSystem:
    def __init__(self, n_agents=100, principles=None, use_median_threshold=False,
                 fitness_criterion='berman', selection_percentile=50, seed=42,
                 invocation_tendencies=None):
        np.random.seed(seed)
        self.principles = principles or PRINCIPLES_DEFAULT
        self.use_median_threshold = use_median_threshold
        self.fitness_criterion = fitness_criterion
        self.selection_percentile = selection_percentile
        self.n_agents = n_agents
        tend = invocation_tendencies or INVOCATION_TENDENCIES
        methods = (METHODOLOGIES*(n_agents//len(METHODOLOGIES)+1))[:n_agents]
        np.random.shuffle(methods)
        self.agents = [Agent.create(i,m,tend) for i,m in enumerate(methods)]
        self.invocation_counts = {p:0 for p in self.principles}
        self.total_invocations = 0
        self.system_weights = {p:0.5 for p in self.principles}
        self.force_outcome_log = []
    def _compute_weights(self):
        if self.total_invocations == 0: return {p:0.5 for p in self.principles}
        raw = {p: self.invocation_counts[p]/self.total_invocations for p in self.principles}
        mx = max(raw.values()) or 1.0
        norm = {p: v/mx for p,v in raw.items()}
        if self.use_median_threshold:
            cutoff = np.percentile(list(norm.values()), self.selection_percentile)
            norm = {p:(v if v>=cutoff else 0.0) for p,v in norm.items()}
            mx2 = max(norm.values()) or 1.0
            norm = {p:v/mx2 for p,v in norm.items()}
        return norm
    def _apply_selection(self):
        rated = sorted(self.agents, key=lambda a: a.success_count/max(a.total_decisions,1))
        n_rep = max(1, self.n_agents//5)
        tops = rated[-n_rep:]
        for agent in rated[:n_rep]:
            tmpl = np.random.choice(tops)
            agent.methodology = tmpl.methodology
            for p in agent.invocation_tendencies:
                agent.invocation_tendencies[p] = np.clip(tmpl.invocation_tendencies[p]+np.random.normal(0,0.08),0.02,0.98)
            agent.success_count = 0; agent.total_decisions = 0
    def simulate(self, n_cases=5000, panel_size=9, update_every=100, selection_every=500):
        for c in range(n_cases):
            activations = {p:np.random.uniform(-1,1) for p in self.principles}
            panel = np.random.choice(self.agents, size=panel_size, replace=False)
            decisions = []
            for agent in panel:
                invoked, force = agent.decide_case(activations, self.system_weights)
                for p in invoked:
                    self.invocation_counts[p] += 1; self.total_invocations += 1
                decisions.append((invoked, force, agent))
            forces = [f for _,f,_ in decisions]
            mean_f = float(np.mean(forces))
            self.force_outcome_log.append((mean_f, abs(mean_f)>0.05))
            if self.fitness_criterion == 'berman':
                med = float(np.median([abs(f) for f in forces]))
                for inv,f,ag in decisions:
                    s = abs(f)>=med; ag.total_decisions+=1
                    if s: ag.success_count+=1
                    ag.update_tendencies(inv,s)
            else:
                pos=sum(1 for f in forces if f>0.05); neg=sum(1 for f in forces if f<-0.05)
                con='pos' if pos>neg else ('neg' if neg>pos else 'tie')
                for inv,f,ag in decisions:
                    d='pos' if f>0.05 else ('neg' if f<-0.05 else 'tie')
                    s=(d==con); ag.total_decisions+=1
                    if s: ag.success_count+=1
                    ag.update_tendencies(inv,s)
            if (c+1)%update_every==0: self.system_weights=self._compute_weights()
            if (c+1)%selection_every==0 and c>0: self._apply_selection()
        w = self.system_weights; wv=np.array(list(w.values()))
        cv_w = float(np.std(wv)/np.mean(wv)) if np.mean(wv)>0 else 0
        return {'system_weights':w, 'cv_weights':cv_w,
                'n_survivors':int(np.sum(wv>0.01)),
                'force_outcome_log':self.force_outcome_log}

print("Core model and ABM loaded.")

Core model and ABM loaded.


## MARL: Strategic Optimisation

Each MARL agent is a small neural network that takes case activations as input
and outputs a probability of invoking each principle. Agents learn through
REINFORCE (policy gradient): they receive a reward based on consensus with the
panel majority, and update their networks to increase the probability of
rewarded invocation patterns.

The key difference from ABM: agents *learn* rather than *evolve*. Bad strategies
improve through gradient descent rather than being replaced by copies of
successful agents. This models rational judges who adapt their reasoning based
on outcomes rather than a cultural process of selection and imitation.

**Architecture:** 9-dimensional input (case activations) → 32-unit hidden layer
→ 9-dimensional output (invocation probabilities). Simple enough that the
theoretical dynamics are transparent.

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Bernoulli

# MARL SYSTEM
# ============================================================================

class JudicialPolicyNetwork(nn.Module):
    """Small policy network: activations → invocation probabilities."""
    def __init__(self, n_principles=9, hidden=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_principles, hidden),
            nn.ReLU(),
            nn.Linear(hidden, n_principles),
            nn.Sigmoid()
        )
    def forward(self, activations):
        return self.net(activations)


class MARLSystem:
    def __init__(self, n_agents=50, n_principles=9, lr=0.003,
                 parsimony_penalty=0.15, seed=42):
        torch.manual_seed(seed)
        np.random.seed(seed)
        self.n_agents = n_agents
        self.n_principles = n_principles
        self.parsimony_penalty = parsimony_penalty
        self.principles = PRINCIPLES_DEFAULT[:n_principles]

        # Create diverse agents with methodology biases in initialisation
        self.networks = []
        self.optimizers = []
        methods = (METHODOLOGIES * (n_agents // len(METHODOLOGIES) + 1))[:n_agents]
        np.random.shuffle(methods)
        self.methodologies = methods

        for i, method in enumerate(methods):
            net = JudicialPolicyNetwork(n_principles)
            # Bias initialisation toward methodology
            tend = INVOCATION_TENDENCIES[method]
            with torch.no_grad():
                bias_vals = torch.tensor([tend.get(p, 0.5) for p in self.principles],
                                         dtype=torch.float32)
                # Shift the output bias so initial probs match tendencies
                net.net[-2].bias.data = torch.logit(torch.clamp(bias_vals, 0.05, 0.95))
            self.networks.append(net)
            self.optimizers.append(optim.Adam(net.parameters(), lr=lr))

    def train(self, n_episodes=3000, panel_size=9, print_every=500):
        history = []

        for ep in range(n_episodes):
            # Generate case
            activations = torch.tensor(
                np.random.uniform(-1, 1, self.n_principles),
                dtype=torch.float32)

            # Select panel
            panel_idx = np.random.choice(self.n_agents, panel_size, replace=False)

            # Each agent decides
            log_probs_list = []
            forces = []
            n_invoked_list = []

            for idx in panel_idx:
                net = self.networks[idx]
                probs = net(activations)
                dist = Bernoulli(probs)
                invoked = dist.sample()
                log_prob = dist.log_prob(invoked).sum()

                # Net force = sum of (system_weight * activation) for invoked principles
                # During training, use invocation probability as weight proxy
                force = (invoked * activations).sum()

                log_probs_list.append(log_prob)
                forces.append(force)
                n_invoked_list.append(invoked.sum().item())

            # Consensus reward: did your force direction match majority?
            force_vals = torch.stack(forces)
            signs = torch.sign(force_vals)
            majority_sign = torch.sign(signs.sum())

            for i, idx in enumerate(panel_idx):
                # Base reward: consensus agreement
                agreed = (signs[i] == majority_sign).float()
                reward = agreed

                # Parsimony: penalise invoking too many principles
                n_inv = n_invoked_list[i]
                parsimony = -self.parsimony_penalty * (n_inv / self.n_principles)
                reward = reward + parsimony

                # Bonus for decisive force
                force_magnitude = abs(force_vals[i].item())
                if force_magnitude > 0.5:
                    reward = reward + 0.1

                # Policy gradient update
                loss = -log_probs_list[i] * reward
                self.optimizers[idx].zero_grad()
                loss.backward()
                self.optimizers[idx].step()

            if (ep + 1) % print_every == 0 or ep == 0:
                weights = self._extract_weights()
                w_vals = np.array(list(weights.values()))
                cv = float(np.std(w_vals) / np.mean(w_vals)) if np.mean(w_vals) > 0 else 0
                mean_inv = np.mean(n_invoked_list)
                history.append({
                    'episode': ep + 1,
                    'weights': weights.copy(),
                    'cv': cv,
                    'mean_invoked': mean_inv,
                })
                print(f"  Episode {ep+1:5d}: CV={cv:.3f}  "
                      f"mean_invoked={mean_inv:.1f}  "
                      f"top={max(w_vals):.3f}  bot={min(w_vals):.3f}")

        return self._summarize(history)

    def _extract_weights(self):
        """Extract principle weights from learned invocation frequencies."""
        n_test = 200
        invoke_counts = np.zeros(self.n_principles)

        with torch.no_grad():
            for _ in range(n_test):
                acts = torch.tensor(np.random.uniform(-1, 1, self.n_principles),
                                    dtype=torch.float32)
                for net in self.networks:
                    probs = net(acts)
                    invoked = (probs > 0.5).float()
                    invoke_counts += invoked.numpy()

        total = invoke_counts.sum()
        if total == 0:
            return {p: 0.5 for p in self.principles}
        raw = invoke_counts / total
        mx = raw.max() or 1.0
        norm = raw / mx
        return {p: float(norm[i]) for i, p in enumerate(self.principles)}

    def _summarize(self, history):
        weights = self._extract_weights()
        w_vals = np.array(list(weights.values()))
        cv = float(np.std(w_vals) / np.mean(w_vals)) if np.mean(w_vals) > 0 else 0

        # K-means gap
        km = KMeans(n_clusters=2, random_state=0, n_init=10)
        labels = km.fit_predict(w_vals.reshape(-1, 1))
        centers = km.cluster_centers_.flatten()
        hi_idx = int(centers[1] > centers[0])
        hi = w_vals[labels == hi_idx]
        lo = w_vals[labels != hi_idx]
        gap = float(hi.min() - lo.max()) if len(hi) > 0 and len(lo) > 0 else 0.0

        # Threshold estimate: median of weights
        threshold = float(np.median(w_vals))

        return {
            'system_weights': weights,
            'cv_weights': cv,
            'gap': gap,
            'threshold': threshold,
            'n_survivors': int(np.sum(w_vals > 0.1)),
            'history': history,
        }

print("MARL system loaded.")
print(f"  Device: {device}")

MARL system loaded.
  Device: cuda


## Hybrid: Neural Learning + Evolutionary Selection

The decisive test. Agents learn via policy gradient (like MARL) but the
worst 20% are periodically replaced with mutated copies of the best
performers (like ABM). This tests whether combining mechanisms produces
better weights or whether the mechanisms interfere.

From prior work, the prediction is that neural learning will erase
evolutionary selection effects — agents that are "culled" and replaced
quickly re-learn the same uniform strategy through gradient descent.

In [5]:
# HYBRID SYSTEM
# ============================================================================

class HybridSystem:
    def __init__(self, n_agents=50, n_principles=9, lr=0.003,
                 parsimony_penalty=0.15,
                 selection_interval=500, replacement_fraction=0.2,
                 mutation_rate=0.1, seed=42):
        torch.manual_seed(seed)
        np.random.seed(seed)
        self.n_agents = n_agents
        self.n_principles = n_principles
        self.parsimony_penalty = parsimony_penalty
        self.selection_interval = selection_interval
        self.replacement_fraction = replacement_fraction
        self.mutation_rate = mutation_rate
        self.principles = PRINCIPLES_DEFAULT[:n_principles]

        methods = (METHODOLOGIES * (n_agents // len(METHODOLOGIES) + 1))[:n_agents]
        np.random.shuffle(methods)

        self.networks = []
        self.optimizers = []
        self.fitness = np.zeros(n_agents)
        self.decisions_made = np.zeros(n_agents)

        for i, method in enumerate(methods):
            net = JudicialPolicyNetwork(n_principles)
            tend = INVOCATION_TENDENCIES[method]
            with torch.no_grad():
                bias_vals = torch.tensor([tend.get(p, 0.5) for p in self.principles],
                                         dtype=torch.float32)
                net.net[-2].bias.data = torch.logit(torch.clamp(bias_vals, 0.05, 0.95))
            self.networks.append(net)
            self.optimizers.append(optim.Adam(net.parameters(), lr=lr))

    def _selection_round(self):
        """Replace worst performers with mutated copies of best."""
        n_rep = max(1, int(self.n_agents * self.replacement_fraction))
        fitness_rates = np.array([
            self.fitness[i] / max(self.decisions_made[i], 1)
            for i in range(self.n_agents)])
        worst_idx = np.argsort(fitness_rates)[:n_rep]
        best_idx = np.argsort(fitness_rates)[-n_rep:]

        for w_i in worst_idx:
            b_i = np.random.choice(best_idx)
            # Copy network weights with mutation
            with torch.no_grad():
                for p_w, p_b in zip(self.networks[w_i].parameters(),
                                     self.networks[b_i].parameters()):
                    p_w.data = p_b.data.clone()
                    p_w.data += torch.randn_like(p_w.data) * self.mutation_rate
            # Reset optimizer
            self.optimizers[w_i] = optim.Adam(self.networks[w_i].parameters(), lr=0.003)
            self.fitness[w_i] = 0
            self.decisions_made[w_i] = 0

        # Reset fitness for all (new epoch)
        best_fit = fitness_rates[best_idx[0]]
        worst_fit = fitness_rates[worst_idx[0]]
        return best_fit, worst_fit

    def train(self, n_episodes=3000, panel_size=9, print_every=500):
        history = []

        for ep in range(n_episodes):
            activations = torch.tensor(
                np.random.uniform(-1, 1, self.n_principles),
                dtype=torch.float32)
            panel_idx = np.random.choice(self.n_agents, panel_size, replace=False)

            log_probs_list = []
            forces = []
            n_invoked_list = []

            for idx in panel_idx:
                net = self.networks[idx]
                probs = net(activations)
                dist = Bernoulli(probs)
                invoked = dist.sample()
                log_prob = dist.log_prob(invoked).sum()
                force = (invoked * activations).sum()
                log_probs_list.append(log_prob)
                forces.append(force)
                n_invoked_list.append(invoked.sum().item())

            force_vals = torch.stack(forces)
            signs = torch.sign(force_vals)
            majority_sign = torch.sign(signs.sum())

            for i, idx in enumerate(panel_idx):
                agreed = (signs[i] == majority_sign).float()
                reward = agreed
                n_inv = n_invoked_list[i]
                reward += -self.parsimony_penalty * (n_inv / self.n_principles)
                if abs(force_vals[i].item()) > 0.5:
                    reward += 0.1

                # Track fitness for selection
                self.fitness[idx] += reward.item()
                self.decisions_made[idx] += 1

                loss = -log_probs_list[i] * reward
                self.optimizers[idx].zero_grad()
                loss.backward()
                self.optimizers[idx].step()

            # Periodic selection
            if (ep + 1) % self.selection_interval == 0 and ep > 0:
                best_f, worst_f = self._selection_round()
                if (ep + 1) % print_every == 0:
                    print(f"  [Episode {ep+1}] SELECTION: best={best_f:.3f} worst={worst_f:.3f}")

            if (ep + 1) % print_every == 0:
                weights = self._extract_weights()
                wv = np.array(list(weights.values()))
                cv = float(np.std(wv)/np.mean(wv)) if np.mean(wv)>0 else 0
                history.append({'episode': ep+1, 'weights': weights, 'cv': cv})
                print(f"  Episode {ep+1:5d}: CV={cv:.3f}  "
                      f"top={max(wv):.3f} bot={min(wv):.3f}")

        return self._summarize(history)

    def _extract_weights(self):
        n_test = 200
        invoke_counts = np.zeros(self.n_principles)
        with torch.no_grad():
            for _ in range(n_test):
                acts = torch.tensor(np.random.uniform(-1,1,self.n_principles),
                                    dtype=torch.float32)
                for net in self.networks:
                    probs = net(acts)
                    invoked = (probs > 0.5).float()
                    invoke_counts += invoked.numpy()
        total = invoke_counts.sum()
        if total == 0: return {p:0.5 for p in self.principles}
        raw = invoke_counts / total
        mx = raw.max() or 1.0
        return {p: float(raw[i]/mx) for i,p in enumerate(self.principles)}

    def _summarize(self, history):
        weights = self._extract_weights()
        wv = np.array(list(weights.values()))
        cv = float(np.std(wv)/np.mean(wv)) if np.mean(wv)>0 else 0
        km = KMeans(n_clusters=2, random_state=0, n_init=10)
        labels = km.fit_predict(wv.reshape(-1,1))
        centers = km.cluster_centers_.flatten()
        hi_idx = int(centers[1]>centers[0])
        hi = wv[labels==hi_idx]; lo = wv[labels!=hi_idx]
        gap = float(hi.min()-lo.max()) if len(hi)>0 and len(lo)>0 else 0.0
        return {'system_weights':weights,'cv_weights':cv,'gap':gap,
                'threshold':float(np.median(wv)),
                'n_survivors':int(np.sum(wv>0.1)),'history':history}

print("Hybrid system loaded.")

Hybrid system loaded.


## Experiment 1 — Three Mechanisms Side by Side

Run ABM, MARL, and Hybrid with the same principle set, then compare the
emergent weight structures. This is a descriptive comparison — which mechanism
produces what kind of weights?

In [6]:
# EXPERIMENT 1: THREE MECHANISMS COMPARED
# ============================================================================

def experiment_1_three_mechanisms(n_seeds=5, abm_cases=5000, marl_episodes=3000):
    """
    Run all three mechanisms and compare emergent weight structures.
    Fewer seeds than Notebooks 0-1 because MARL is expensive.
    """
    print("\n" + "=" * 70)
    print("EXPERIMENT 1: Three Mechanisms Side by Side")
    print(f"  Seeds: {n_seeds}")
    print(f"  ABM: {abm_cases} cases | MARL: {marl_episodes} episodes")
    print("=" * 70)

    results = {'abm': [], 'marl': [], 'hybrid': []}

    for seed in range(n_seeds):
        print(f"\n  --- Seed {seed} ---")

        # ABM (Condition C: Berman + threshold)
        print(f"  Running ABM...")
        lsys = LegalSystem(use_median_threshold=True,
                           fitness_criterion='berman', seed=seed)
        r_abm = lsys.simulate(n_cases=abm_cases)
        results['abm'].append(r_abm)
        print(f"    ABM: CV={r_abm['cv_weights']:.3f}  survivors={r_abm['n_survivors']}")

        # MARL
        print(f"  Running MARL...")
        marl = MARLSystem(n_agents=50, seed=seed)
        r_marl = marl.train(n_episodes=marl_episodes, print_every=marl_episodes)
        results['marl'].append(r_marl)
        print(f"    MARL: CV={r_marl['cv_weights']:.3f}  survivors={r_marl['n_survivors']}")

        # Hybrid
        print(f"  Running Hybrid...")
        hybrid = HybridSystem(n_agents=50, seed=seed)
        r_hybrid = hybrid.train(n_episodes=marl_episodes, print_every=marl_episodes)
        results['hybrid'].append(r_hybrid)
        print(f"    Hybrid: CV={r_hybrid['cv_weights']:.3f}  survivors={r_hybrid['n_survivors']}")

    # Summary table
    print(f"\n  {'='*65}")
    print(f"  {'Metric':<25} {'ABM':>12} {'MARL':>12} {'Hybrid':>12}")
    print(f"  {'-'*65}")

    for metric_name, key in [('Weight CV', 'cv_weights'),
                              ('N survivors (w>0.1)', 'n_survivors')]:
        vals = {}
        for mech in ['abm', 'marl', 'hybrid']:
            v = [r[key] for r in results[mech]]
            vals[mech] = np.mean(v)
        print(f"  {metric_name:<25} {vals['abm']:>12.3f} {vals['marl']:>12.3f} {vals['hybrid']:>12.3f}")

    # Weight gap
    for mech in ['abm', 'marl', 'hybrid']:
        if 'gap' in results[mech][0]:
            gaps = [r['gap'] for r in results[mech]]
            results[mech + '_gap_mean'] = np.mean(gaps)

    # Print representative weights for each
    for mech in ['abm', 'marl', 'hybrid']:
        w = results[mech][0]['system_weights']
        sorted_w = sorted(w.items(), key=lambda x: x[1], reverse=True)
        top3 = ', '.join(f"{p}={v:.3f}" for p,v in sorted_w[:3])
        bot3 = ', '.join(f"{p}={v:.3f}" for p,v in sorted_w[-3:])
        print(f"\n  {mech.upper()} weights (seed 0):")
        print(f"    Top 3: {top3}")
        print(f"    Bot 3: {bot3}")

    return results

exp1_results = experiment_1_three_mechanisms(n_seeds=10)


EXPERIMENT 1: Three Mechanisms Side by Side
  Seeds: 10
  ABM: 5000 cases | MARL: 3000 episodes

  --- Seed 0 ---
  Running ABM...
    ABM: CV=0.896  survivors=5
  Running MARL...
  Episode     1: CV=0.424  mean_invoked=3.8  top=1.000  bot=0.048
  Episode  3000: CV=0.341  mean_invoked=3.8  top=1.000  bot=0.322
    MARL: CV=0.342  survivors=9
  Running Hybrid...
  [Episode 3000] SELECTION: best=0.915 worst=0.830
  Episode  3000: CV=1.260  top=1.000 bot=0.000
    Hybrid: CV=1.257  survivors=4

  --- Seed 1 ---
  Running ABM...
    ABM: CV=0.896  survivors=5
  Running MARL...
  Episode     1: CV=0.439  mean_invoked=4.0  top=1.000  bot=0.032
  Episode  3000: CV=0.350  mean_invoked=3.2  top=1.000  bot=0.280
    MARL: CV=0.347  survivors=9
  Running Hybrid...
  [Episode 3000] SELECTION: best=0.933 worst=0.845
  Episode  3000: CV=1.262  top=1.000 bot=0.000
    Hybrid: CV=1.264  survivors=4

  --- Seed 2 ---
  Running ABM...
    ABM: CV=0.895  survivors=5
  Running MARL...
  Episode     1: CV

In [7]:
# EXPERIMENT 1 — VISUALISATION
# ============================================================================

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Experiment 1: Three Mechanisms — Weight Structures',
             fontsize=14, fontweight='bold')

mechanisms = ['abm', 'marl', 'hybrid']
colors_mech = {'abm': 'steelblue', 'marl': 'coral', 'hybrid': 'mediumpurple'}

# Top row: Weight profiles (seed 0) for each mechanism
for col, mech in enumerate(mechanisms):
    ax = axes[0, col]
    w = exp1_results[mech][0]['system_weights']
    sorted_w = sorted(w.items(), key=lambda x: x[1], reverse=True)
    names = [x[0] for x in sorted_w]
    vals = [x[1] for x in sorted_w]
    bar_colors = ['#2ca02c' if v > 0.1 else '#d62728' for v in vals]
    ax.barh(range(len(names)), vals, color=bar_colors, alpha=0.8, edgecolor='black')
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=9)
    cv = exp1_results[mech][0]['cv_weights']
    surv = exp1_results[mech][0]['n_survivors']
    ax.set_title(f'{mech.upper()}\nCV={cv:.3f}, survivors={surv}', fontweight='bold')
    ax.set_xlim(0, 1.1)
    ax.grid(True, alpha=0.3, axis='x')

# Bottom left: CV distribution across seeds
ax = axes[1, 0]
for mech in mechanisms:
    cvs = [r['cv_weights'] for r in exp1_results[mech]]
    positions = [mechanisms.index(mech)]
    bp = ax.boxplot([cvs], positions=positions, widths=0.5,
                    patch_artist=True)
    bp['boxes'][0].set_facecolor(colors_mech[mech])
    bp['boxes'][0].set_alpha(0.7)
ax.set_xticks(range(len(mechanisms)))
ax.set_xticklabels([m.upper() for m in mechanisms])
ax.set_ylabel('Weight CV')
ax.set_title('Weight Differentiation\n(across seeds)', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Bottom middle: Survivors distribution
ax = axes[1, 1]
for j, mech in enumerate(mechanisms):
    survs = [r['n_survivors'] for r in exp1_results[mech]]
    ax.bar(j, np.mean(survs), yerr=np.std(survs),
           color=colors_mech[mech], alpha=0.8, edgecolor='black', capsize=5)
ax.set_xticks(range(len(mechanisms)))
ax.set_xticklabels([m.upper() for m in mechanisms])
ax.set_ylabel('N surviving principles (w > 0.1)')
ax.set_title('How Many Principles Survive?', fontweight='bold')
ax.set_ylim(0, 10)
ax.grid(True, alpha=0.3, axis='y')

# Bottom right: MARL weight evolution (from history if available)
ax = axes[1, 2]
if exp1_results['marl'][0].get('history'):
    hist = exp1_results['marl'][0]['history']
    episodes = [h['episode'] for h in hist]
    for p in PRINCIPLES_DEFAULT:
        vals = [h['weights'].get(p, 0) for h in hist]
        ax.plot(episodes, vals, linewidth=1.5, label=p)
    ax.set_xlabel('Episode')
    ax.set_ylabel('Weight')
    ax.set_title('MARL Weight Evolution\n(Seed 0)', fontweight='bold')
    ax.legend(fontsize=6, ncol=2)
    ax.grid(True, alpha=0.3)
else:
    ax.text(0.5, 0.5, 'No MARL history\navailable', transform=ax.transAxes,
            ha='center', va='center', fontsize=12)
    ax.set_title('MARL Weight Evolution', fontweight='bold')

plt.tight_layout()
plt.savefig('/content/exp1_mechanisms.png')
plt.show()

## Experiment 2 — MARL Diagnostics

The MARL may produce uniform weights because of the consensus reward structure.
We test this by varying the parsimony penalty — the only lever that pushes
agents toward selectivity. If strong parsimony produces differentiation,
the reward structure is the bottleneck. If not, strategic optimisation
fundamentally resists differentiation.

In [8]:
# EXPERIMENT 2: MARL DIAGNOSTICS
# ============================================================================

def experiment_2_marl_diagnostics(n_seeds=3, n_episodes=2000):
    print("\n" + "=" * 70)
    print("EXPERIMENT 2: MARL Diagnostics — Parsimony Sweep")
    print("=" * 70)

    penalties = [0.0, 0.15, 0.50, 1.0, 2.0]
    rows = []

    for penalty in penalties:
        cvs = []; gaps = []
        for seed in range(n_seeds):
            marl = MARLSystem(n_agents=50, parsimony_penalty=penalty, seed=seed)
            r = marl.train(n_episodes=n_episodes, print_every=n_episodes+1)
            cvs.append(r['cv_weights'])
            gaps.append(r['gap'])
        m_cv, lo_cv, hi_cv = bootstrap_ci(cvs)
        m_gap = np.mean(gaps)
        rows.append({'penalty': penalty, 'cv_mean': m_cv, 'cv_lo': lo_cv,
                     'cv_hi': hi_cv, 'gap_mean': m_gap})
        print(f"  Parsimony={penalty:.2f}: CV={m_cv:.3f} [{lo_cv:.3f}, {hi_cv:.3f}]  "
              f"gap={m_gap:.3f}")

    df = pd.DataFrame(rows)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle('MARL Diagnostics: Does Parsimony Create Differentiation?',
                 fontsize=13, fontweight='bold')
    ax1.errorbar(df['penalty'], df['cv_mean'],
                 yerr=[df['cv_mean']-df['cv_lo'], df['cv_hi']-df['cv_mean']],
                 marker='o', capsize=5, linewidth=2)
    ax1.set_xlabel('Parsimony penalty')
    ax1.set_ylabel('Weight CV')
    ax1.set_title('Differentiation vs parsimony')
    ax1.grid(True, alpha=0.3)

    ax2.bar(range(len(penalties)), df['gap_mean'], color='steelblue',
            edgecolor='black', alpha=0.8)
    ax2.set_xticks(range(len(penalties)))
    ax2.set_xticklabels([f'{p:.2f}' for p in penalties])
    ax2.set_xlabel('Parsimony penalty')
    ax2.set_ylabel('K-means gap')
    ax2.set_title('Bimodal gap vs parsimony')
    ax2.grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.savefig('/content/exp2_marl_diagnostics.png')
    plt.show()

    return df

exp2_df = experiment_2_marl_diagnostics()


EXPERIMENT 2: MARL Diagnostics — Parsimony Sweep
  Episode     1: CV=0.424  mean_invoked=3.8  top=1.000  bot=0.048
  Episode     1: CV=0.439  mean_invoked=4.0  top=1.000  bot=0.032
  Episode     1: CV=0.430  mean_invoked=3.8  top=1.000  bot=0.054
  Parsimony=0.00: CV=0.352 [0.344, 0.359]  gap=0.267
  Episode     1: CV=0.424  mean_invoked=3.8  top=1.000  bot=0.048
  Episode     1: CV=0.439  mean_invoked=4.0  top=1.000  bot=0.032
  Episode     1: CV=0.430  mean_invoked=3.8  top=1.000  bot=0.054
  Parsimony=0.15: CV=0.361 [0.347, 0.372]  gap=0.258
  Episode     1: CV=0.424  mean_invoked=3.8  top=1.000  bot=0.048
  Episode     1: CV=0.439  mean_invoked=4.0  top=1.000  bot=0.032
  Episode     1: CV=0.430  mean_invoked=3.8  top=1.000  bot=0.054
  Parsimony=0.50: CV=0.382 [0.373, 0.390]  gap=0.203
  Episode     1: CV=0.424  mean_invoked=3.8  top=1.000  bot=0.048
  Episode     1: CV=0.439  mean_invoked=4.0  top=1.000  bot=0.032
  Episode     1: CV=0.430  mean_invoked=3.8  top=1.000  bot=0.054

### Experiment 2b — MARL Architecture Sensitivity

A reviewer will ask whether the MARL result depends on the network being too
simple. We test three architectures:
- **Small:** 1 layer, 32 hidden units (current default)
- **Medium:** 2 layers, 64 hidden units each
- **Large:** 2 layers, 128 hidden units each

If all three produce similar weight structures, the finding is about strategic
optimisation, not network capacity. If the larger network produces
differentiation, the single-layer result was an artifact.

In [9]:
# EXPERIMENT 2b: MARL ARCHITECTURE SENSITIVITY
# ============================================================================

class JudicialPolicyNetworkDeep(nn.Module):
    """Deeper policy network for architecture sensitivity test."""
    def __init__(self, n_principles=9, hidden=64, n_layers=2):
        super().__init__()
        layers = [nn.Linear(n_principles, hidden), nn.ReLU()]
        for _ in range(n_layers - 1):
            layers.extend([nn.Linear(hidden, hidden), nn.ReLU()])
        layers.extend([nn.Linear(hidden, n_principles), nn.Sigmoid()])
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)


class MARLSystemConfigurable(MARLSystem):
    """MARLSystem with configurable network architecture."""
    def __init__(self, n_agents=50, n_principles=9, lr=0.003,
                 parsimony_penalty=0.15, seed=42,
                 hidden=32, n_layers=1):
        # Don't call super().__init__ — rebuild with custom network
        torch.manual_seed(seed)
        np.random.seed(seed)
        self.n_agents = n_agents
        self.n_principles = n_principles
        self.parsimony_penalty = parsimony_penalty
        self.principles = PRINCIPLES_DEFAULT[:n_principles]

        methods = (METHODOLOGIES * (n_agents // len(METHODOLOGIES) + 1))[:n_agents]
        np.random.shuffle(methods)
        self.methodologies = methods

        self.networks = []
        self.optimizers = []
        for i, method in enumerate(methods):
            if n_layers == 1:
                net = JudicialPolicyNetwork(n_principles, hidden)
            else:
                net = JudicialPolicyNetworkDeep(n_principles, hidden, n_layers)
            tend = INVOCATION_TENDENCIES[method]
            with torch.no_grad():
                bias_vals = torch.tensor([tend.get(p, 0.5) for p in self.principles],
                                         dtype=torch.float32)
                # Find last linear layer's bias
                for module in reversed(list(net.modules())):
                    if isinstance(module, nn.Linear) and module.out_features == n_principles:
                        module.bias.data = torch.logit(torch.clamp(bias_vals, 0.05, 0.95))
                        break
            self.networks.append(net)
            self.optimizers.append(optim.Adam(net.parameters(), lr=lr))


def experiment_2b_architecture(n_seeds=5, n_episodes=2000):
    print("\n" + "=" * 70)
    print("EXPERIMENT 2b: MARL Architecture Sensitivity")
    print("  Does network capacity change the result?")
    print("=" * 70)

    configs = [
        ('1×32',  32, 1),
        ('2×64',  64, 2),
        ('2×128', 128, 2),
    ]

    rows = []
    for label, hidden, n_layers in configs:
        cvs = []; survivors = []; all_weights = []
        for seed in range(n_seeds):
            marl = MARLSystemConfigurable(
                n_agents=50, hidden=hidden, n_layers=n_layers,
                seed=seed)
            r = marl.train(n_episodes=n_episodes, print_every=n_episodes+1)
            cvs.append(r['cv_weights'])
            survivors.append(r['n_survivors'])
            all_weights.append(r['system_weights'])

        m_cv, lo_cv, hi_cv = bootstrap_ci(cvs)
        m_surv = np.mean(survivors)
        rows.append({'arch': label, 'cv_mean': m_cv, 'cv_lo': lo_cv,
                     'cv_hi': hi_cv, 'survivors': m_surv})
        print(f"  {label:6s}: CV={m_cv:.3f} [{lo_cv:.3f}, {hi_cv:.3f}]  "
              f"survivors={m_surv:.1f}")

    df = pd.DataFrame(rows)

    # Statistical test: is 2×128 significantly different from 1×32?
    cvs_small = []
    cvs_large = []
    for seed in range(n_seeds):
        m1 = MARLSystemConfigurable(n_agents=50, hidden=32, n_layers=1, seed=seed)
        r1 = m1.train(n_episodes=n_episodes, print_every=n_episodes+1)
        cvs_small.append(r1['cv_weights'])
        m2 = MARLSystemConfigurable(n_agents=50, hidden=128, n_layers=2, seed=seed)
        r2 = m2.train(n_episodes=n_episodes, print_every=n_episodes+1)
        cvs_large.append(r2['cv_weights'])

    stat, pval = stats.wilcoxon(cvs_small, cvs_large)
    diff = np.mean(cvs_large) - np.mean(cvs_small)
    print(f"\n  1×32 vs 2×128: ΔCV = {diff:+.3f}, p = {pval:.4f}")
    if pval < 0.05:
        print(f"  → Significant difference. Architecture matters.")
    else:
        print(f"  → No significant difference. Result is about strategic")
        print(f"    optimisation, not network capacity.")

    return df

exp2b_df = experiment_2b_architecture(n_seeds=5)


EXPERIMENT 2b: MARL Architecture Sensitivity
  Does network capacity change the result?
  Episode     1: CV=0.424  mean_invoked=3.8  top=1.000  bot=0.048
  Episode     1: CV=0.439  mean_invoked=4.0  top=1.000  bot=0.032
  Episode     1: CV=0.430  mean_invoked=3.8  top=1.000  bot=0.054
  Episode     1: CV=0.415  mean_invoked=4.1  top=1.000  bot=0.069
  Episode     1: CV=0.428  mean_invoked=4.2  top=1.000  bot=0.052
  1×32  : CV=0.358 [0.347, 0.369]  survivors=9.0
  Episode     1: CV=0.449  mean_invoked=4.0  top=1.000  bot=0.002
  Episode     1: CV=0.456  mean_invoked=4.8  top=1.000  bot=0.000
  Episode     1: CV=0.439  mean_invoked=3.9  top=1.000  bot=0.002
  Episode     1: CV=0.455  mean_invoked=3.9  top=1.000  bot=0.000
  Episode     1: CV=0.449  mean_invoked=4.1  top=1.000  bot=0.000
  2×64  : CV=0.338 [0.326, 0.354]  survivors=9.0
  Episode     1: CV=0.448  mean_invoked=3.0  top=1.000  bot=0.000
  Episode     1: CV=0.453  mean_invoked=4.3  top=1.000  bot=0.002
  Episode     1: CV=0

In [10]:
# EXPERIMENT 2b — VISUALISATION
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(range(len(exp2b_df)), exp2b_df['cv_mean'],
       yerr=[exp2b_df['cv_mean'] - exp2b_df['cv_lo'],
             exp2b_df['cv_hi'] - exp2b_df['cv_mean']],
       color=['steelblue', 'coral', 'mediumpurple'][:len(exp2b_df)],
       alpha=0.8, edgecolor='black', capsize=5)
ax.set_xticks(range(len(exp2b_df)))
ax.set_xticklabels(exp2b_df['arch'].values, fontsize=10)
ax.set_xlabel('Architecture')
ax.set_ylabel('Weight CV')
ax.set_title('MARL Architecture Sensitivity\n(Does network capacity matter?)',
             fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('/content/exp2b_architecture.png')
plt.show()

## Experiment 3 — Cross-Validation: Which Criteria Does Each Mechanism Pass?

This is the paper's main result. We run all three sets of emergent weights
through Notebook 0's five criteria and produce the comparison table showing
that ABM and MARL have complementary failure patterns.

In [11]:
# EXPERIMENT 3: CROSS-VALIDATION
# ============================================================================

def crossvalidate_weights(label, weights_dict, force_log=None):
    """Run Notebook 0's five criteria on a set of emergent weights."""
    w = weights_dict
    p_names = list(w.keys())
    w_vals = np.array(list(w.values()))

    results = {}

    # C1: Weight uncertainty tolerance >= 0.07
    live = [(p, wt) for p, wt in w.items() if wt > 0.001]
    if len(live) < 2:
        results['C1'] = {'passed': False, 'value': 0.0}
    else:
        principles_em = [Principle(p, float(wt)) for p, wt in live]
        w_arr = np.array([wt for _, wt in live])
        n_p = len(live)
        rng = np.random.RandomState(42)
        n_test, n_mc = 300, 40
        sigma_levels = np.linspace(0, 0.50, 16)
        a_mat = rng.uniform(-1, 1, size=(n_test, n_p))
        noise = rng.normal(0, 1, size=(n_test, n_mc, n_p))
        true_det = np.abs(a_mat @ w_arr) >= 0.2
        curve = [1.0]
        for sigma in sigma_levels[1:]:
            nw = np.clip(w_arr[None,None,:] + sigma*noise, 0, 1)
            nf = np.sum(nw * a_mat[:,None,:], axis=2)
            nd = np.abs(nf) >= 0.2
            agree = np.mean(nd == true_det[:,None], axis=1)
            curve.append(float(np.mean(agree >= 0.9)))
        crit = find_crossing(np.array(curve), sigma_levels, 0.80)
        results['C1'] = {'passed': crit >= 0.07, 'value': crit}

    # C2: Berman beats Hart >= +19.1%
    if len(live) >= 2:
        rng2 = np.random.RandomState(123)
        n_h = 500
        b_det, h_det = 0, 0
        sorted_p = sorted(w.items(), key=lambda x: x[1], reverse=True)
        tier1 = {p for p,_ in sorted_p[:2]}
        tier2 = {p for p,_ in sorted_p[2:5]}
        for _ in range(n_h):
            acts = {p: rng2.uniform(-1,1) for p in p_names}
            fb = sum(w[p]*acts[p] for p in p_names)
            if abs(fb) >= 0.2: b_det += 1
            t1s = any(abs(acts[p])>0.7 for p in tier1)
            t2s = any(abs(acts[p])>0.7 for p in tier2)
            t1m = any(abs(acts[p])>0.4 for p in tier1)
            if t1s or (t1m and t2s): h_det += 1
        adv = b_det/n_h - h_det/n_h
        results['C2'] = {'passed': adv >= 0.191, 'value': adv}
    else:
        results['C2'] = {'passed': False, 'value': 0.0}

    # C3: Threshold in 0.05-0.30
    if force_log:
        det_f = [abs(f) for f,d in force_log if d]
        ind_f = [abs(f) for f,d in force_log if not d]
        if det_f and ind_f:
            thresh = (np.percentile(det_f, 5) + np.percentile(ind_f, 95)) / 2
        else:
            thresh = np.median(w_vals)
    else:
        thresh = float(np.median(w_vals))
    results['C3'] = {'passed': 0.05 <= thresh <= 0.30, 'value': thresh}

    # C4: Correlation helps
    if len(live) >= 2:
        rng3 = np.random.RandomState(456)
        n_ct, n_mc_c = 200, 30
        sigma_t = 0.10
        noise_c = rng3.normal(0, sigma_t, size=(n_ct, n_mc_c, n_p))
        nw_c = np.clip(w_arr[None,None,:]+noise_c, 0, 1)

        a_ind = rng3.uniform(-1,1,size=(n_ct, n_p))
        td_ind = np.abs(a_ind @ w_arr) >= 0.2
        nf_ind = np.sum(nw_c * a_ind[:,None,:], axis=2)
        ag_ind = float(np.mean(np.mean((np.abs(nf_ind)>=0.2)==td_ind[:,None],axis=1)>=0.9))

        half = n_p//2
        f1 = rng3.uniform(-1,1,size=(n_ct,1)); f2 = rng3.uniform(-1,1,size=(n_ct,1))
        a_corr = np.zeros((n_ct,n_p))
        a_corr[:,:half] = np.clip(0.5*f1+0.5*rng3.uniform(-1,1,size=(n_ct,half)),-1,1)
        a_corr[:,half:] = np.clip(0.5*f2+0.5*rng3.uniform(-1,1,size=(n_ct,n_p-half)),-1,1)
        td_corr = np.abs(a_corr @ w_arr) >= 0.2
        nf_corr = np.sum(nw_c * a_corr[:,None,:], axis=2)
        ag_corr = float(np.mean(np.mean((np.abs(nf_corr)>=0.2)==td_corr[:,None],axis=1)>=0.9))
        results['C4'] = {'passed': ag_corr >= ag_ind, 'value': ag_corr - ag_ind}
    else:
        results['C4'] = {'passed': False, 'value': 0.0}

    # C5: Cardinal variation (not binary, not uniform)
    live_w = w_vals[w_vals > 0.01]
    if len(live_w) >= 2:
        cv = float(np.std(live_w)/np.mean(live_w))
        is_binary = bool(np.all((live_w > 0.9) | (live_w < 0.1)))
        is_uniform = cv < 0.05
        results['C5'] = {'passed': not is_binary and not is_uniform and cv > 0.05,
                          'value': cv}
    else:
        results['C5'] = {'passed': False, 'value': 0.0}

    return results


def experiment_3_crossvalidation(exp1_results, n_seeds=10):
    print("\n" + "=" * 70)
    print("EXPERIMENT 3: Cross-Validation — All Mechanisms vs Notebook 0 Criteria")
    print("=" * 70)

    criteria_names = {
        'C1': 'Weight uncertainty ≥ ±0.07',
        'C2': 'Berman beats Hart ≥ +19.1%',
        'C3': 'Threshold in 0.05–0.30',
        'C4': 'Correlation helps robustness',
        'C5': 'Cardinal weight variation',
    }

    all_results = {}
    for mech in ['abm', 'marl', 'hybrid']:
        print(f"\n  --- {mech.upper()} ---")
        mech_results = {c: [] for c in criteria_names}

        for seed_idx in range(min(n_seeds, len(exp1_results[mech]))):
            r = exp1_results[mech][seed_idx]
            w = r['system_weights']
            fl = r.get('force_outcome_log', None)
            cv_result = crossvalidate_weights(mech, w, fl)

            for c, res in cv_result.items():
                mech_results[c].append(res['passed'])

            # Print seed details
            vals = {c: cv_result[c]['value'] for c in criteria_names}
            statuses = ' '.join(
                f"{'✅' if cv_result[c]['passed'] else '❌'}"
                for c in sorted(criteria_names))
            print(f"    Seed {seed_idx}: {statuses}")

        all_results[mech] = mech_results

    # MAIN RESULT TABLE
    print(f"\n  {'='*75}")
    print(f"  MAIN RESULT: Criteria Pass Rates by Mechanism")
    print(f"  {'='*75}")
    print(f"  {'Criterion':<35} {'ABM':>10} {'MARL':>10} {'Hybrid':>10}")
    print(f"  {'-'*75}")

    for c in sorted(criteria_names):
        rates = {}
        for mech in ['abm', 'marl', 'hybrid']:
            vals = all_results[mech][c]
            rate = sum(vals) / len(vals) if vals else 0
            rates[mech] = rate
        status_abm = '✅' if rates['abm'] >= 0.8 else ('⚠️' if rates['abm'] >= 0.4 else '❌')
        status_marl = '✅' if rates['marl'] >= 0.8 else ('⚠️' if rates['marl'] >= 0.4 else '❌')
        status_hyb = '✅' if rates['hybrid'] >= 0.8 else ('⚠️' if rates['hybrid'] >= 0.4 else '❌')
        print(f"  {status_abm} {status_marl} {status_hyb}  {criteria_names[c]:<30} "
              f"{rates['abm']:>8.0%} {rates['marl']:>10.0%} {rates['hybrid']:>10.0%}")

    # Complementarity analysis
    print(f"\n  COMPLEMENTARITY ANALYSIS:")
    abm_passes = {c for c in criteria_names if sum(all_results['abm'][c])/len(all_results['abm'][c]) >= 0.8}
    marl_passes = {c for c in criteria_names if sum(all_results['marl'][c])/max(len(all_results['marl'][c]),1) >= 0.8}
    union = abm_passes | marl_passes
    intersection = abm_passes & marl_passes
    abm_only = abm_passes - marl_passes
    marl_only = marl_passes - abm_passes
    neither = set(criteria_names.keys()) - union

    print(f"    ABM passes:  {', '.join(sorted(abm_passes)) or 'none'}")
    print(f"    MARL passes: {', '.join(sorted(marl_passes)) or 'none'}")
    print(f"    Both pass:   {', '.join(sorted(intersection)) or 'none'}")
    print(f"    ABM only:    {', '.join(sorted(abm_only)) or 'none'}")
    print(f"    MARL only:   {', '.join(sorted(marl_only)) or 'none'}")
    print(f"    Neither:     {', '.join(sorted(neither)) or 'none'}")

    if abm_only and marl_only:
        print(f"\n  → COMPLEMENTARY FAILURE PATTERNS CONFIRMED.")
        print(f"    ABM gets {', '.join(sorted(abm_only))} right but not {', '.join(sorted(marl_only))}.")
        print(f"    MARL gets {', '.join(sorted(marl_only))} right but not {', '.join(sorted(abm_only))}.")
        print(f"    No single mechanism satisfies all theoretical requirements.")
    elif union == set(criteria_names.keys()):
        print(f"\n  → Together they cover everything, but neither alone suffices.")
    else:
        print(f"\n  → {', '.join(sorted(neither))} not satisfied by either mechanism.")

    return all_results

exp3_results = experiment_3_crossvalidation(exp1_results)


EXPERIMENT 3: Cross-Validation — All Mechanisms vs Notebook 0 Criteria

  --- ABM ---
    Seed 0: ✅ ❌ ✅ ✅ ❌
    Seed 1: ✅ ❌ ✅ ✅ ❌
    Seed 2: ✅ ❌ ✅ ✅ ❌
    Seed 3: ✅ ❌ ✅ ✅ ❌
    Seed 4: ✅ ❌ ✅ ✅ ❌
    Seed 5: ✅ ❌ ✅ ✅ ❌
    Seed 6: ✅ ❌ ✅ ✅ ❌
    Seed 7: ✅ ❌ ✅ ✅ ❌
    Seed 8: ✅ ❌ ✅ ✅ ❌
    Seed 9: ✅ ❌ ✅ ✅ ❌

  --- MARL ---
    Seed 0: ✅ ❌ ❌ ✅ ✅
    Seed 1: ✅ ❌ ❌ ✅ ✅
    Seed 2: ✅ ❌ ❌ ✅ ✅
    Seed 3: ✅ ❌ ❌ ✅ ✅
    Seed 4: ✅ ❌ ❌ ✅ ✅
    Seed 5: ✅ ❌ ❌ ✅ ✅
    Seed 6: ✅ ❌ ❌ ✅ ✅
    Seed 7: ✅ ❌ ❌ ✅ ✅
    Seed 8: ✅ ❌ ❌ ✅ ✅
    Seed 9: ✅ ❌ ❌ ✅ ✅

  --- HYBRID ---
    Seed 0: ✅ ❌ ❌ ✅ ✅
    Seed 1: ✅ ❌ ✅ ✅ ✅
    Seed 2: ✅ ❌ ✅ ✅ ✅
    Seed 3: ✅ ❌ ❌ ✅ ✅
    Seed 4: ✅ ❌ ✅ ✅ ✅
    Seed 5: ✅ ❌ ✅ ✅ ✅
    Seed 6: ✅ ❌ ✅ ✅ ✅
    Seed 7: ✅ ❌ ✅ ✅ ✅
    Seed 8: ✅ ❌ ✅ ✅ ✅
    Seed 9: ✅ ❌ ❌ ✅ ✅

  MAIN RESULT: Criteria Pass Rates by Mechanism
  Criterion                                  ABM       MARL     Hybrid
  ---------------------------------------------------------------------------
  ✅ ✅ ✅  Weight uncer

### Experiment 3b — Full Berman-vs-Hart Comparison with Emergent Weights

The C2 criterion (Berman beats Hart by ≥19.1%) was calibrated against
hierarchical Hart with stipulated weights. But that is one comparison point.
The complete picture requires running Berman against all Hart variants
(consensus at 50%–90% plus hierarchical) using emergent weights from each
mechanism.

This is not goalpost-moving. The C2 pass/fail result stands as reported.
This experiment shows the full Berman advantage curve so the paper can state
precisely: "With emergent weights, Berman beats consensus Hart by X% but
the advantage against hierarchical Hart narrows to Y% because both favour
textual principles."

In [12]:
# EXPERIMENT 3b: FULL BERMAN-VS-HART WITH EMERGENT WEIGHTS
# ============================================================================

def experiment_3b_full_hart(exp1_results, n_test=1000):
    """
    Run Notebook 0's Experiment 4 (Berman vs multiple Hart variants)
    using emergent weights from each mechanism.
    """
    print("\n" + "=" * 70)
    print("EXPERIMENT 3b: Full Berman-vs-Hart with Emergent Weights")
    print("  How does the advantage change with different Hart models?")
    print("=" * 70)

    hart_consensus_thresholds = [0.50, 0.60, 0.70, 0.80, 0.90]

    all_advantages = {}

    for mech in ['abm', 'marl', 'hybrid']:
        advantages = {k: [] for k in hart_consensus_thresholds + ['hierarchical']}

        for seed_idx in range(len(exp1_results[mech])):
            r = exp1_results[mech][seed_idx]
            w = r['system_weights']
            p_names = list(w.keys())

            # Hart hierarchical tiers (top 2 weights = tier 1)
            sorted_p = sorted(w.items(), key=lambda x: x[1], reverse=True)
            tier1 = {p for p, _ in sorted_p[:2]}
            tier2 = {p for p, _ in sorted_p[2:5]}

            rng = np.random.RandomState(seed_idx + 7000)

            berman_det = 0
            hart_results = {k: 0 for k in hart_consensus_thresholds}
            hart_hier = 0

            for _ in range(n_test):
                acts = {p: rng.uniform(-1, 1) for p in p_names}

                # Berman
                f_b = sum(w[p] * acts[p] for p in p_names)
                if abs(f_b) >= 0.2:
                    berman_det += 1

                # Consensus Hart variants
                pos = sum(1 for p in p_names if acts[p] > 0.2)
                neg = sum(1 for p in p_names if acts[p] < -0.2)
                total = len(p_names)
                for k in hart_consensus_thresholds:
                    if pos / total >= k or neg / total >= k:
                        hart_results[k] += 1

                # Hierarchical Hart
                t1s = any(abs(acts[p]) > 0.7 for p in tier1)
                t2s = any(abs(acts[p]) > 0.7 for p in tier2)
                t1m = any(abs(acts[p]) > 0.4 for p in tier1)
                if t1s or (t1m and t2s):
                    hart_hier += 1

            b_rate = berman_det / n_test
            for k in hart_consensus_thresholds:
                advantages[k].append(b_rate - hart_results[k] / n_test)
            advantages['hierarchical'].append(b_rate - hart_hier / n_test)

        all_advantages[mech] = advantages

    # Report
    print(f"\n  {'Hart variant':<20}", end="")
    for mech in ['abm', 'marl', 'hybrid']:
        print(f"  {mech.upper():>12}", end="")
    print()
    print(f"  {'-'*60}")

    for variant in hart_consensus_thresholds + ['hierarchical']:
        label = f"Consensus {int(variant*100)}%" if isinstance(variant, float) else "Hierarchical"
        print(f"  {label:<20}", end="")
        for mech in ['abm', 'marl', 'hybrid']:
            vals = all_advantages[mech][variant]
            m = np.mean(vals)
            print(f"  {m:>+11.1%}", end="")
        print()

    # The key finding
    print(f"\n  KEY FINDING:")
    for mech in ['abm', 'marl', 'hybrid']:
        cons50 = np.mean(all_advantages[mech][0.50])
        hier = np.mean(all_advantages[mech]['hierarchical'])
        print(f"    {mech.upper():7s}: vs consensus-50% = {cons50:+.1%},  "
              f"vs hierarchical = {hier:+.1%}")

    print(f"\n  INTERPRETATION:")
    print(f"  Berman still beats consensus Hart with emergent weights.")
    print(f"  The advantage narrows against hierarchical Hart because")
    print(f"  emergent weights (like Hart's hierarchy) favour textual principles.")
    print(f"  C2's failure at ≥19.1% is specific to the hierarchical comparison,")
    print(f"  not a general failure of Berman's advantage.")

    # Visualise
    fig, ax = plt.subplots(figsize=(10, 6))
    fig.suptitle('Berman Advantage Over Hart: Stipulated vs Emergent Weights',
                 fontsize=13, fontweight='bold')

    colors = {'abm': 'steelblue', 'marl': 'coral', 'hybrid': 'mediumpurple'}
    x_labels = [f'{int(k*100)}%' for k in hart_consensus_thresholds] + ['Hier.']
    x = np.arange(len(x_labels))

    for mech in ['abm', 'marl', 'hybrid']:
        means = [np.mean(all_advantages[mech][k]) for k in hart_consensus_thresholds]
        means.append(np.mean(all_advantages[mech]['hierarchical']))
        ax.plot(x, means, 'o-', color=colors[mech], linewidth=2,
                markersize=8, label=f'{mech.upper()} weights')

    ax.axhline(0, color='black', linewidth=0.5)
    ax.axhline(0.191, color='red', linestyle='--', linewidth=1.5,
               label='Notebook 0 C2 threshold (19.1%)')
    ax.set_xticks(x)
    ax.set_xticklabels(x_labels)
    ax.set_xlabel('Hart variant')
    ax.set_ylabel('Berman advantage (proportion)')
    ax.set_title('Does Berman still beat Hart with emergent weights?')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('/content/exp3b_hart_curve.png')
    plt.show()

    return all_advantages

exp3b_advantages = experiment_3b_full_hart(exp1_results)


EXPERIMENT 3b: Full Berman-vs-Hart with Emergent Weights
  How does the advantage change with different Hart models?

  Hart variant                   ABM          MARL        HYBRID
  ------------------------------------------------------------
  Consensus 50%              +35.1%       +33.8%       +32.5%
  Consensus 60%              +68.0%       +66.7%       +65.4%
  Consensus 70%              +82.8%       +81.5%       +80.2%
  Consensus 80%              +87.2%       +85.9%       +84.6%
  Consensus 90%              +88.0%       +86.7%       +85.4%
  Hierarchical               +15.2%       +13.9%       +12.7%

  KEY FINDING:
    ABM    : vs consensus-50% = +35.1%,  vs hierarchical = +15.2%
    MARL   : vs consensus-50% = +33.8%,  vs hierarchical = +13.9%
    HYBRID : vs consensus-50% = +32.5%,  vs hierarchical = +12.7%

  INTERPRETATION:
  Berman still beats consensus Hart with emergent weights.
  The advantage narrows against hierarchical Hart because
  emergent weights (like Hart's

## Experiment 4 — Summary Visualisation

Publication-quality figure showing the three mechanisms' weight structures
and criteria pass/fail patterns.

In [13]:
# EXPERIMENT 4: SUMMARY VISUALISATION
# ============================================================================

fig = plt.figure(figsize=(18, 10))
fig.suptitle('Mechanism Dependence: Three Practice Models, Complementary Failures',
             fontsize=14, fontweight='bold')

# Top row: Weight profiles for each mechanism (seed 0)
for col, mech in enumerate(['abm', 'marl', 'hybrid']):
    ax = fig.add_subplot(2, 3, col + 1)
    w = exp1_results[mech][0]['system_weights']
    sorted_w = sorted(w.items(), key=lambda x: x[1], reverse=True)
    names = [x[0] for x in sorted_w]
    vals = [x[1] for x in sorted_w]
    colors = ['#2ca02c' if v > 0.1 else '#d62728' for v in vals]
    ax.barh(range(len(names)), vals, color=colors, alpha=0.8, edgecolor='black')
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=8)
    cv = exp1_results[mech][0]['cv_weights']
    ax.set_title(f'{mech.upper()}\nCV={cv:.3f}', fontweight='bold')
    ax.set_xlim(0, 1.1)
    ax.grid(True, alpha=0.3, axis='x')

# Bottom left: Criteria pass/fail heatmap
ax = fig.add_subplot(2, 3, 4)
criteria_order = ['C1', 'C2', 'C3', 'C4', 'C5']
criteria_labels = ['C1: Weight\ntolerance', 'C2: Beats\nHart',
                   'C3: Threshold\nrange', 'C4: Corr\nhelps', 'C5: Cardinal\nvariation']
mechs = ['abm', 'marl', 'hybrid']
heatmap_data = np.zeros((len(criteria_order), len(mechs)))
for j, mech in enumerate(mechs):
    for i, c in enumerate(criteria_order):
        vals = exp3_results[mech][c]
        heatmap_data[i, j] = sum(vals) / len(vals) if vals else 0

im = ax.imshow(heatmap_data, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(len(mechs)))
ax.set_xticklabels([m.upper() for m in mechs], fontsize=10)
ax.set_yticks(range(len(criteria_order)))
ax.set_yticklabels(criteria_labels, fontsize=8)
ax.set_title('Criteria Pass Rates', fontweight='bold')
for i in range(len(criteria_order)):
    for j in range(len(mechs)):
        val = heatmap_data[i, j]
        txt = f'{val:.0%}'
        color = 'white' if val < 0.4 or val > 0.8 else 'black'
        ax.text(j, i, txt, ha='center', va='center', fontsize=9,
                fontweight='bold', color=color)
plt.colorbar(im, ax=ax, shrink=0.8)

# Bottom middle: CV comparison
ax = fig.add_subplot(2, 3, 5)
for mech in mechs:
    cvs = [r['cv_weights'] for r in exp1_results[mech]]
    m, lo, hi = bootstrap_ci(cvs)
    idx = mechs.index(mech)
    ax.bar(idx, m, yerr=[[m-lo], [hi-m]], color=['steelblue','coral','mediumpurple'][idx],
           alpha=0.8, edgecolor='black', capsize=5)
ax.set_xticks(range(len(mechs)))
ax.set_xticklabels([m.upper() for m in mechs])
ax.set_ylabel('Weight CV')
ax.set_title('Weight Differentiation', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Bottom right: Text summary
ax = fig.add_subplot(2, 3, 6)
ax.axis('off')
summary_text = (
    "KEY FINDING:\n\n"
    "No single practice mechanism\n"
    "passes all five criteria.\n\n"
    "ABM (evolutionary):\n"
    "  ✅ Differentiation (legal vs sub-legal)\n"
    "  ❌ No gradation among survivors\n\n"
    "MARL (strategic):\n"
    "  ✅ Graded weights (cardinal variation)\n"
    "  ❌ No differentiation (all principles similar)\n\n"
    "Hybrid:\n"
    "  ❌ Learning erases selection\n"
    "  ❌ Worst of both mechanisms\n\n"
    "→ Practice-grounding requires\n"
    "   specifying the mechanism."
)
ax.text(0.1, 0.95, summary_text, transform=ax.transAxes,
        fontsize=10, verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.savefig('/content/mechanism_dependence_summary.png')
plt.show()

## Summary and Interpretation

### The main finding

No single practice mechanism produces weights that satisfy all five of
Notebook 0's theoretical requirements simultaneously. This is not a failure
of the computational approach — it is a finding about the gap between
Berman's theory and its practice-grounded implementation.

**ABM (evolutionary):** Passes C1 (weight tolerance), C3 (threshold range),
C4 (correlation helps). Fails C2 (Hart advantage) and C5 (cardinal variation).
Produces binary differentiation — principles are either fully legal or dead.

**MARL (strategic):** Passes C4 (correlation helps) and C5 (cardinal variation).
Fails C1 (weight tolerance), C2 (Hart advantage), C3 (threshold range).
Produces graded but undifferentiated weights — all principles survive at
moderate levels.

**Hybrid:** Results are sensitive to the balance between learning and selection.
In our implementation, selection dominates, producing ABM-like weights with
the addition of cardinal variation (passing C1, C4, C5). But the instability
across implementations makes it theoretically unreliable.

### C2: The universal failure

No mechanism produces weights where Berman beats hierarchical Hart by the
≥19.1% threshold from Notebook 0. However, Experiment 3b shows that Berman
*does* still beat consensus Hart variants with emergent weights. The
hierarchical advantage narrows because both Berman's emergent weights and
Hart's hierarchy favour textual principles — a consequence of bench
composition, not a flaw in Berman's aggregation mechanism.

The paper should report this honestly: Berman's advantage over Hart is
real but mechanism-dependent and comparison-dependent. Against consensus
models of Hart, the advantage is substantial. Against hierarchical models,
it narrows when weights are practice-emergent rather than stipulated.

### What this means for Berman's theory

Berman says parameters are "practice-grounded." The computational evidence
confirms that parameters *can* emerge from practice, but:

1. The mechanism of practice determines the weight structure.
2. Evolutionary practice produces differentiation but not gradation.
3. Strategic practice produces gradation but not differentiation.
4. No single mechanism we tested produces the full set of properties
   the theory requires.

This suggests that the gradation of weights among surviving legal principles
may require something beyond bare evolutionary or strategic dynamics —
perhaps the kind of explicit judicial reasoning about relative importance
that Berman describes qualitatively but which simple agents cannot model.

### Limitations

- MARL uses small neural networks (tested up to 2×128). Larger or
  recurrent architectures might behave differently.
- The ABM's methodology structure is fixed. Different distributions of
  judicial methodologies would produce different emergent weights.
- Five to ten seeds per mechanism. Sufficient for qualitative findings
  but not for precise parameter estimation.
- The Hybrid result is sensitive to implementation choices and should
  be interpreted cautiously.

In [15]:
# COMPREHENSIVE EXPORT — ALL RESULTS
# ============================================================================

import json, os, shutil
out = "/content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Berman Modeling/Experiment/Computational /Outputs/2"
os.makedirs(out, exist_ok=True)
print(f"Exporting all Notebook 2 results to:\n  {out}\n")

# =====================================================================
# EXPERIMENT 1: Three Mechanisms — Weights
# =====================================================================
for mech in ['abm', 'marl', 'hybrid']:
    rows = []
    for seed, r in enumerate(exp1_results[mech]):
        row = {'seed': seed, 'cv_weights': r['cv_weights'],
               'n_survivors': r['n_survivors']}
        row.update(r['system_weights'])
        rows.append(row)
    pd.DataFrame(rows).to_csv(f'{out}/exp1_{mech}_weights.csv', index=False)
    print(f"  exp1_{mech}_weights.csv")

# Summary
exp1_summary_rows = []
for mech in ['abm', 'marl', 'hybrid']:
    cvs = [r['cv_weights'] for r in exp1_results[mech]]
    survs = [r['n_survivors'] for r in exp1_results[mech]]
    m_cv, lo_cv, hi_cv = bootstrap_ci(cvs)
    exp1_summary_rows.append({
        'mechanism': mech,
        'cv_mean': m_cv, 'cv_lo': lo_cv, 'cv_hi': hi_cv,
        'survivors_mean': np.mean(survs), 'survivors_std': np.std(survs),
        'n_seeds': len(cvs),
    })
pd.DataFrame(exp1_summary_rows).to_csv(f'{out}/exp1_summary.csv', index=False)
print("  exp1_summary.csv")

# =====================================================================
# EXPERIMENT 2: Parsimony Sweep
# =====================================================================
exp2_df.to_csv(f'{out}/exp2_parsimony.csv', index=False)
print("  exp2_parsimony.csv")

# =====================================================================
# EXPERIMENT 2b: Architecture Sensitivity
# =====================================================================
exp2b_df.to_csv(f'{out}/exp2b_architecture.csv', index=False)
print("  exp2b_architecture.csv")

# =====================================================================
# EXPERIMENT 3: Cross-Validation — Per Seed
# =====================================================================
exp3_rows = []
for mech in ['abm', 'marl', 'hybrid']:
    for c, vals in exp3_results[mech].items():
        for seed, passed in enumerate(vals):
            exp3_rows.append({'mechanism': mech, 'criterion': c,
                              'seed': seed, 'passed': int(passed)})
pd.DataFrame(exp3_rows).to_csv(f'{out}/exp3_crossvalidation.csv', index=False)
print("  exp3_crossvalidation.csv")

# Pass rate summary
exp3_summary_rows = []
for mech in ['abm', 'marl', 'hybrid']:
    for c in ['C1', 'C2', 'C3', 'C4', 'C5']:
        vals = exp3_results[mech][c]
        rate = sum(vals)/len(vals) if vals else 0
        exp3_summary_rows.append({
            'mechanism': mech, 'criterion': c,
            'pass_rate': rate, 'n_pass': sum(vals), 'n_total': len(vals),
        })
pd.DataFrame(exp3_summary_rows).to_csv(f'{out}/exp3_pass_rates.csv', index=False)
print("  exp3_pass_rates.csv")

# =====================================================================
# EXPERIMENT 3b: Hart Curve with Emergent Weights
# =====================================================================
exp3b_rows = []
for mech in ['abm', 'marl', 'hybrid']:
    for variant in [0.50, 0.60, 0.70, 0.80, 0.90, 'hierarchical']:
        vals = exp3b_advantages[mech][variant]
        m = np.mean(vals)
        lo = np.percentile(vals, 2.5) if len(vals) > 1 else m
        hi = np.percentile(vals, 97.5) if len(vals) > 1 else m
        label = f'consensus_{int(variant*100)}' if isinstance(variant, float) else 'hierarchical'
        exp3b_rows.append({
            'mechanism': mech, 'hart_variant': label,
            'advantage_mean': m, 'advantage_lo': lo, 'advantage_hi': hi,
            'n_seeds': len(vals),
        })
pd.DataFrame(exp3b_rows).to_csv(f'{out}/exp3b_hart_curve.csv', index=False)
print("  exp3b_hart_curve.csv")

# =====================================================================
# ALL PLOTS
# =====================================================================
plot_dir = f'{out}/plots'
os.makedirs(plot_dir, exist_ok=True)
for png in [f for f in os.listdir('/content') if f.endswith('.png')]:
    shutil.copy2(f'/content/{png}', f'{plot_dir}/{png}')
    print(f"  plots/{png}")

# =====================================================================
# MASTER SUMMARY JSON
# =====================================================================
summary = {
    'experiment_1': {
        mech: {
            'cv_mean': float(np.mean([r['cv_weights'] for r in exp1_results[mech]])),
            'cv_ci': list(bootstrap_ci([r['cv_weights'] for r in exp1_results[mech]])),
            'survivors_mean': float(np.mean([r['n_survivors'] for r in exp1_results[mech]])),
            'n_seeds': len(exp1_results[mech]),
        } for mech in ['abm', 'marl', 'hybrid']
    },
    'experiment_3_pass_rates': {
        mech: {
            c: float(sum(exp3_results[mech][c])/len(exp3_results[mech][c]))
            for c in ['C1', 'C2', 'C3', 'C4', 'C5']
        } for mech in ['abm', 'marl', 'hybrid']
    },
    'experiment_3b_hart_advantages': {
        mech: {
            'consensus_50': float(np.mean(exp3b_advantages[mech][0.50])),
            'consensus_70': float(np.mean(exp3b_advantages[mech][0.70])),
            'hierarchical': float(np.mean(exp3b_advantages[mech]['hierarchical'])),
        } for mech in ['abm', 'marl', 'hybrid']
    },
    'complementarity': {
        'abm_passes': sorted([c for c in ['C1','C2','C3','C4','C5']
                               if sum(exp3_results['abm'][c])/len(exp3_results['abm'][c]) >= 0.8]),
        'marl_passes': sorted([c for c in ['C1','C2','C3','C4','C5']
                                if sum(exp3_results['marl'][c])/len(exp3_results['marl'][c]) >= 0.8]),
        'hybrid_passes': sorted([c for c in ['C1','C2','C3','C4','C5']
                                  if sum(exp3_results['hybrid'][c])/len(exp3_results['hybrid'][c]) >= 0.8]),
    },
}

with open(f'{out}/notebook2_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print("  notebook2_summary.json")

all_files = []
for root, dirs, files in os.walk(out):
    for fname in files:
        all_files.append(os.path.join(root, fname))
print(f"\n{'='*50}")
print(f"TOTAL: {len(all_files)} files exported to {out}")
print(f"{'='*50}")

Exporting all Notebook 2 results to:
  /content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Berman Modeling/Experiment/Computational /Outputs/2

  exp1_abm_weights.csv
  exp1_marl_weights.csv
  exp1_hybrid_weights.csv
  exp1_summary.csv
  exp2_parsimony.csv
  exp2b_architecture.csv
  exp3_crossvalidation.csv
  exp3_pass_rates.csv
  exp3b_hart_curve.csv
  plots/exp2_marl_diagnostics.png
  plots/exp1_mechanisms.png
  plots/exp3b_hart_curve.png
  plots/exp2b_architecture.png
  plots/mechanism_dependence_summary.png
  notebook2_summary.json

TOTAL: 15 files exported to /content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Berman Modeling/Experiment/Computational /Outputs/2
